In [1]:
!pip install pydantic

In [2]:
!pip install langchain-google-genai

In [3]:
!pip install -qU "langchain[google-genai]"

In [4]:
import os
from typing import Final
from telegram import Update
from telegram.ext import ContextTypes
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

In [5]:
load_dotenv()
TELEGRAM_BOT_TOKEN: Final = os.getenv("TOKEN")
TELEGRAM_BOT_USERNAME: Final = os.getenv("BOT_USERNAME")
API_KEY = os.getenv("GEMINI_API_KEY")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import LLMChain
# Instantiate Gemini model with config
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # latest identifier
    google_api_key= API_KEY,
    temperature=0.7,
    max_tokens=1024,
)
#create prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly, knowledgeable ecommerce assistant named Sahayak. Your role is to help users discover, compare, and understand products available online, especially budget-friendly options in India. You speak clearly and concisely, using everyday language. Provide helpful suggestions and answer questions related to categories like fashion, electronics, home goods, and deals. Always guide users politely, and if you don't know something or lack real-time access, recommend searching online. Never give financial or medical advice. Use emojis sparingly to make responses more engaging. Stay factual and avoid personal opinions."),
    ("human", "{input}")
])
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=300,  # Customize based on your model's context window
    memory_key="chat_history",
    input_key="input",
    summary_prompt="summarize the user and his/her preferences from past messages"
)
chain = LLMChain(llm=llm, prompt=prompt, memory=memory)
# Async function to interact with Gemini using LangChain
def askAi(content: str) -> dict:
    response = chain.invoke(input=content)
    print(memory.chat_memory.messages)
    return response["text"]

C:\Users\Yash Sinha\AppData\Local\Temp\ipykernel_20672\3052522568.py:17: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(
C:\Users\Yash Sinha\AppData\Local\Temp\ipykernel_20672\3052522568.py:24: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt, memory=memory)


##Commands

In [7]:
async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """
    Sends a greeting message when the /start command is issued.

    Args:
        update (Update): Incoming update from Telegram.
        context (ContextTypes.DEFAULT_TYPE): Context object with metadata and bot data.
    """
    await update.message.reply_text(
        "Hello, give me a description of the product you want to search and let's get going!!!!"
    )

In [8]:
async def help_command (update: Update, context: ContextTypes.DEFAULT_TYPE):
    """sends a message to user when confused

    Args:
        update (Update): _description_
        context (ContextTypes.DEFAULT_TYPE): _description_
    """
    await update.message.reply_text("Give me a description of the product you want to buy to get started.")

##Responses

In [9]:
def handle_response (text: str) -> str:
    """handle responses

    Args:
        text (str): _description_

    Returns:
        str: _description_
    """
    response = askAi(text)
    # print('Our Response: ', response)
    return response

In [10]:
async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """handling messages

    Args:
        update (Update): _description_
        context (ContextTypes.DEFAULT_TYPE): _description_
    """
    message_type: str = update.message.chat.type
    text: str = update.message.text
    
    print(f'user ({update.message.chat.id}) in {message_type}: "{text}"')
    
    if message_type == 'group':
        if TELEGRAM_BOT_USERNAME in text:
            new_text: str = text.replace(TELEGRAM_BOT_USERNAME, '').strip()
            response: str = handle_response(new_text)
        else:
            return
    else:
        response: str = handle_response(text)
    
    #print('Bot: ', response)
    
    await update.message.reply_text(response)

In [11]:
async def error(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print(f'Update {update} caused error: {context.error}')
    

In [12]:
from telegram.ext import Application, CommandHandler, MessageHandler, filters
print('starting bot....')
app = Application.builder().token(TELEGRAM_BOT_TOKEN).build()

#commands
app.add_handler(CommandHandler('start', start_command))
app.add_handler(CommandHandler('help', help_command))

#messsages
app.add_handler(MessageHandler(filters.TEXT, handle_message))

#errors
app.add_error_handler(error)

#polls the bot
print("polling...")
app.run_polling(poll_interval=3)

starting bot....
polling...


RuntimeError: Cannot close a running event loop